<a href="https://colab.research.google.com/github/mimiktam/BU_AI_660_Assignments/blob/main/Week2_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Giving Google Colab permission to connect to my personal Google Drive space.

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%%writefile week2_assignment.py

# This is a fully implemented week2_assignment.py file

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier

def load_and_map_data(filepath='credit_risk_dataset.csv'):
    """PHASE 1: ENVIRONMENT SETUP & REAL DATA EDA"""
    # 1. Real Data Ingestion
    try:
        raw_data = pd.read_csv(filepath)
    except FileNotFoundError:
        raise FileNotFoundError(f"Please ensure '{filepath}' is saved in your local directory.")

    # Standardizing raw columns to align with clean, intuitive variable names
    data = raw_data[[
        'loan_int_rate',
        'person_income',
        'person_age',
        'person_home_ownership',
        'loan_intent',
        'loan_status'
    ]].copy()
    data.columns = ['interest_rate', 'annual_income', 'age', 'home_ownership', 'purpose', 'default']
    print("--- Phase 1: Real Data Ingestion & Mapping Complete ---")

    # 2. Target Assessment
    target_counts = data['default'].value_counts()
    target_pct = data['default'].value_counts(normalize=True) * 100
    print(f"Real Target Distribution:\nClass 0 (Non-Default): {target_counts[0]} ({target_pct[0]:.2f}%)")
    print(f"Class 1 (Default):     {target_counts[1]} ({target_pct[1]:.2f}%)\n")

    # 3. Feature Isolation & Stratified Split
    X = data.drop(columns=['default'])
    y = data['default']

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )

    return X_train, X_val, y_train, y_val

def build_preprocessing_architecture(X_train):
    """PHASE 2: PREPROCESSING ARCHITECTURE"""
    # Automatically identifying numerical and categorical features
    numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

    # Defining sub-pipelines (Handling real dataset missing values natively)
    numerical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # Synthesizing into a single ColumnTransformer
    preprocessor = ColumnTransformer(transformers=[
        ('num', numerical_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ])

    print("--- Phase 2: Preprocessing Architecture Formed ---")
    return preprocessor

def train_master_pipeline(preprocessor, X_train, y_train):
    """PHASE 3: MODEL EXECUTION"""
    # Calculating the exact algorithmic guardrail scaling parameter for real imbalanced data
    value_counts = y_train.value_counts()
    negative_cases = value_counts.get(0, 1)
    positive_cases = value_counts.get(1, 1)
    calculated_scale_pos_weight = negative_cases / positive_cases

    print(f"Calculated scale_pos_weight: {calculated_scale_pos_weight:.2f}")

    # Constructing the master Pipeline
    master_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(
            scale_pos_weight=calculated_scale_pos_weight,
            random_state=42,
            eval_metric='logloss'
        ))
    ])

    # Model Fit (Learns coefficients naturally from data)
    master_pipeline.fit(X_train, y_train)
    print("--- Phase 3: Master Pipeline Trained Successfully ---\n")
    return master_pipeline

def run_evaluation_phases(master_pipeline, X_val, y_val):
    """PHASE 4 & 5: STATISTICAL EVALUATION & FINANCIAL IMPACT ANALYSIS
    This runs your threshold optimization simulation and displays your charts.
    """
    # ==========================================
    # PHASE 4: STATISTICAL EVALUATION (Standard 0.50 Threshold)
    # ==========================================
    y_pred_baseline = master_pipeline.predict(X_val)
    y_probs = master_pipeline.predict_proba(X_val)[:, 1]

    print("--- Phase 4: Statistical Metrics (Standard 0.50 Threshold) ---")
    print(classification_report(y_val, y_pred_baseline))
    print(f"ROC-AUC Score: {roc_auc_score(y_val, y_probs):.4f}\n")

    # ==========================================
    # PHASE 5: FINANCIAL IMPACT ANALYSIS
    # ==========================================
    baseline_tco, tn, fp, fn, tp = calculate_tco(y_val, y_pred_baseline)
    print("--- Phase 5: Financial Impact Analysis ---")
    print(f"Baseline TCO at 0.50 Threshold: ${baseline_tco:,}")
    print(f"↳ False Positives (Type I): {fp} | False Negatives (Type II): {fn}\n")

    # Threshold Optimization Analysis
    thresholds = [0.50, 0.40, 0.30, 0.20, 0.15, 0.10, 0.05]
    tco_results = []

    print("Optimizing Decision Thresholds to Mitigate Risk:")
    print("-" * 65)
    print(f"{'Threshold':<12}{'False Pos (FP)':<16}{'False Neg (FN)':<16}{'Total TCO':<15}")
    print("-" * 65)

    for t in thresholds:
        y_pred_custom = (y_probs >= t).astype(int)
        cost, tn_c, fp_c, fn_c, tp_c = calculate_tco(y_val, y_pred_custom)
        tco_results.append((t, cost))
        print(f"{t:<12}{fp_c:<16}{fn_c:<16}${cost:<14,}")

    print("-" * 65)
    best_threshold, min_cost = min(tco_results, key=lambda item: item[1])
    print(f"Optimal Business Decision Threshold discovered at: {best_threshold}")
    print(f"Minimized Enterprise Financial Risk (TCO): ${min_cost:,}")

def calculate_tco(y_true, y_pred_labels):
    """Core financial engine utilized by Phase 5 optimization checks."""
    COST_FP = 500
    COST_FN = 5000

    if len(np.unique(y_true)) == 1 or len(np.unique(y_pred_labels)) == 1:
        cm = confusion_matrix(y_true, y_pred_labels, labels=[0, 1])
    else:
        cm = confusion_matrix(y_true, y_pred_labels)

    tn, fp, fn, tp = cm.ravel()
    tco = (fp * COST_FP) + (fn * COST_FN)
    return tco, tn, fp, fn, tp

# Optional: This blocks allows you to run "python week2_assignment.py"
# directly to see all your output logs instantly, independent of tester.py!
if __name__ == "__main__":
    X_train, X_val, y_train, y_val = load_and_map_data()
    preprocessor = build_preprocessing_architecture(X_train)
    pipeline = train_master_pipeline(preprocessor, X_train, y_train)
    run_evaluation_phases(pipeline, X_val, y_val)




Overwriting week2_assignment.py


In [11]:
!python week2_assignment.py

--- Phase 1: Real Data Ingestion & Mapping Complete ---
Real Target Distribution:
Class 0 (Non-Default): 25473 (78.18%)
Class 1 (Default):     7108 (21.82%)

--- Phase 2: Preprocessing Architecture Formed ---
Calculated scale_pos_weight: 3.58
--- Phase 3: Master Pipeline Trained Successfully ---

--- Phase 4: Statistical Metrics (Standard 0.50 Threshold) ---
              precision    recall  f1-score   support

           0       0.93      0.82      0.87      5095
           1       0.55      0.77      0.64      1422

    accuracy                           0.81      6517
   macro avg       0.74      0.80      0.76      6517
weighted avg       0.84      0.81      0.82      6517

ROC-AUC Score: 0.8850

--- Phase 5: Financial Impact Analysis ---
Baseline TCO at 0.50 Threshold: $2,072,500
↳ False Positives (Type I): 915 | False Negatives (Type II): 323

Optimizing Decision Thresholds to Mitigate Risk:
-----------------------------------------------------------------
Threshold   False Pos 

In [4]:
%%writefile test.py

# test.py
import numpy as np
import pandas as pd

# =====================================================================
# TEST CASES FOR: load_and_map_data()
# =====================================================================
test_cases_load = {
    "case_1_standard": "credit_risk_dataset.csv",
    "case_2_missing_file": "non_existent_file.csv",
    "case_3_corrupted_columns": "corrupted_file.csv",
    "case_4_empty_dataset": "empty_file.csv"
}

# =====================================================================
# TEST CASES FOR: build_preprocessing_architecture()
# =====================================================================
test_cases_preprocess = [
    # Case 1: Ideal dataset entry
    pd.DataFrame({
        'interest_rate': [11.5], 'annual_income': [60000], 'age': [25],
        'home_ownership': ['RENT'], 'purpose': ['EDUCATION']
    }),
    # Case 2: Extreme missing numerical data (Testing the Median Imputer)
    pd.DataFrame({
        'interest_rate': [np.nan], 'annual_income': [45000], 'age': [np.nan],
        'home_ownership': ['MORTGAGE'], 'purpose': ['MEDICAL']
    }),
    # Case 3: Missing categorical entries (Testing the 'missing' string flag)
    pd.DataFrame({
        'interest_rate': [8.5], 'annual_income': [120000], 'age': [42],
        'home_ownership': [np.nan], 'purpose': [np.nan]
    }),
    # Case 4: Unseen structural string category (Testing OneHotEncoder handle_unknown='ignore')
    pd.DataFrame({
        'interest_rate': [15.0], 'annual_income': [30000], 'age': [19],
        'home_ownership': ['SPACE_STATION'], 'purpose': ['INTERSTELLAR_TRAVEL']
    })
]

# =====================================================================
# TEST CASES FOR: train_master_pipeline()
# =====================================================================
test_cases_train = [
    # Case 1: Standard balanced framework
    {"X": pd.DataFrame({'interest_rate': [10, 12], 'annual_income': [50000, 60000], 'age': [30, 40], 'home_ownership': ['OWN', 'RENT'], 'purpose': ['PERSONAL', 'VENTURE']}), "y": pd.Series([0, 1])},
    # Case 2: Highly Imbalanced framework (Testing stability of extreme scale_pos_weight)
    {"X": pd.DataFrame({'interest_rate': [12]*10, 'annual_income': [80000]*10, 'age': [35]*10, 'home_ownership': ['RENT']*10, 'purpose': ['DEBTCONSOLIDATION']*10}), "y": pd.Series([0]*9 + [1])},
    # Case 3: Minimal operational shape matrix
    {"X": pd.DataFrame({'interest_rate': [9], 'annual_income': [100000], 'age': [50], 'home_ownership': ['MORTGAGE'], 'purpose': ['HOMEIMPROVEMENT']}), "y": pd.Series([0])},
    # Case 4: Scale boundary maximum dataset split
    {"X": pd.DataFrame({'interest_rate': [25.0, 5.0], 'annual_income': [10000, 500000], 'age': [18, 90], 'home_ownership': ['RENT', 'OWN'], 'purpose': ['MEDICAL', 'VENTURE']}), "y": pd.Series([1, 0])}
]

# =====================================================================
# TEST CASES FOR: calculate_tco()
# =====================================================================
test_cases_tco = [
    # Case 1: Perfect prediction array ($0 Costs)
    {"y_true": np.array([0, 0, 1, 1]), "y_pred": np.array([0, 0, 1, 1]), "expected_tco": 0},
    # Case 2: pure Type I Errors only (False Positives: 2 * $500 = $1,000)
    {"y_true": np.array([0, 0, 1, 1]), "y_pred": np.array([1, 1, 1, 1]), "expected_tco": 1000},
    # Case 3: Pure Type II Errors only (False Negatives: 2 * $5000 = $10,000)
    {"y_true": np.array([0, 0, 1, 1]), "y_pred": np.array([0, 0, 0, 0]), "expected_tco": 10000},
    # Case 4: Complex mixed real-world error portfolio (1 FP + 1 FN = $5,500)
    {"y_true": np.array([0, 0, 1, 1]), "y_pred": np.array([0, 1, 0, 1]), "expected_tco": 5500}
]


Overwriting test.py


In [13]:
%%writefile tester.py

# tester.py

import sys
import importlib
import inspect
import os

def run_harness():
    # 1. Parse and validate runtime arguments
    if len(sys.argv) < 2:
        print("Error: Missing target execution module.")
        print("Usage: python tester.py <filename.py>")
        return

    target_file = sys.argv[1]
    module_name = target_file.replace('.py', '')

    if not os.path.exists(target_file):
        print(f"Error: Target file '{target_file}' not found.")
        return

    # 2. Safely bootstrap testing modules
    try:
        import test as suite
        target_module = importlib.import_module(module_name)
    except Exception as e:
        print(f"System compilation execution error: {str(e)}")
        return

    print("=" * 70)
    print(f"EXECUTING GRADUATED MLOPS TEST HARNESS ON: {target_file}")
    print("=" * 70)

    # Initialize counter variables for the final tally
    not_implemented_tally = 0
    passed_tally = 0
    failed_tally = 0

    # List target pipeline validation hooks
    target_functions = [
        "load_and_map_data",
        "build_preprocessing_architecture",
        "train_master_pipeline",
        "calculate_tco"
    ]

    for func_name in target_functions:
        print(f"\nChecking Function: {func_name}()")
        print("-" * 50)

        # Check if function exists
        if not hasattr(target_module, func_name):
            print(" -> Result: [Not yet implemented] (Missing function definition)")
            not_implemented_tally += 1
            continue

        func_object = getattr(target_module, func_name)
        source_code = inspect.getsource(func_object)

        # Verify template emptiness via source tracking
        if "pass" in source_code and len(source_code.strip().split('\n')) <= 3:
            print(" -> Result: [Not yet implemented] (Template intact, no logic found)")
            not_implemented_tally += 1
            continue

        # Verify template emptiness via NotImplementedError string check
        if "raise NotImplementedError" in source_code:
            print(" -> Result: [Not yet implemented] (Template intact, explicitly raises exception)")
            not_implemented_tally += 1
            continue

        # 4. Run Assertions against Test Suite Data
        try:
            if func_name == "load_and_map_data":
                # Running integration sanity check (Counts as 1 core test case)
                X_tr, X_v, y_tr, y_v = func_object(suite.test_cases_load["case_1_standard"])
                print(f" [PASS] Successfully loaded partitions. Train shape: {X_tr.shape}")
                passed_tally += 1

            elif func_name == "build_preprocessing_architecture":
                # Evaluating preprocessing pipelines against extreme cases (4 sub-cases)
                for idx, sample_df in enumerate(suite.test_cases_preprocess, 1):
                    try:
                        transformer = func_object(sample_df)
                        transformed_meta = transformer.fit_transform(sample_df)
                        print(f" [PASS] Sub-case {idx} resolved across sparse matrices.")
                        passed_tally += 1
                    except Exception as e:
                        print(f" [FAIL] Sub-case {idx} failed: {str(e)}")
                        failed_tally += 1

            elif func_name == "train_master_pipeline":
                # Evaluating dynamic XGBoost binding pipeline setup (Counts as 1 core case)
                sample_case = suite.test_cases_train[0]
                try:
                    preprocessor = getattr(target_module, "build_preprocessing_architecture")(sample_case["X"])
                    fitted_pipeline = func_object(preprocessor, sample_case["X"], sample_case["y"])
                    from sklearn.pipeline import Pipeline
                    assert isinstance(fitted_pipeline, Pipeline), "Returned object is not a valid Pipeline"
                    print(" [PASS] Production pipeline synthesis bound successfully.")
                    passed_tally += 1
                except Exception as e:
                    print(f" [FAIL] Pipeline binding failed: {str(e)}")
                    failed_tally += 1

            elif func_name == "calculate_tco":
                # Evaluating monetary optimization engine metrics (4 sub-cases)
                for idx, case in enumerate(suite.test_cases_tco, 1):
                    try:
                        calculated_tco, *_ = func_object(case["y_true"], case["y_pred"])
                        assert calculated_tco == case["expected_tco"], f"Cost mismatch on case {idx}"
                        print(f" [PASS] Cost Matrix Verification Sub-case {idx}: ${calculated_tco:,}")
                        passed_tally += 1
                    except Exception as e:
                        print(f" [FAIL] Cost Matrix Verification Sub-case {idx} failed: {str(e)}")
                        failed_tally += 1

            print(f"==> STATUS: {func_name.upper()} PROCESSING COMPLETED.")

        except Exception as eval_err:
            print(f" -> Result: [Critical Structural Failure]: {str(eval_err)}")
            failed_tally += 1

    # 5. Display the Final Metrics Dashboard Tally
    print("\n" + "=" * 70)
    print("FINAL PERFORMANCE TALLY SUMMARY")
    print("=" * 70)
    print(f"Total number of test cases not implemented: {not_implemented_tally}")
    print(f"Total number of test cases passed:          {passed_tally}")
    print(f"Total number of test cases failed:          {failed_tally}")
    print("=" * 70)

if __name__ == "__main__":
    run_harness()

Overwriting tester.py


In [14]:
!python tester.py week2_assignment.py

EXECUTING GRADUATED MLOPS TEST HARNESS ON: week2_assignment.py

Checking Function: load_and_map_data()
--------------------------------------------------
--- Phase 1: Real Data Ingestion & Mapping Complete ---
Real Target Distribution:
Class 0 (Non-Default): 25473 (78.18%)
Class 1 (Default):     7108 (21.82%)

 [PASS] Successfully loaded partitions. Train shape: (26064, 5)
==> STATUS: LOAD_AND_MAP_DATA PROCESSING COMPLETED.

Checking Function: build_preprocessing_architecture()
--------------------------------------------------
--- Phase 2: Preprocessing Architecture Formed ---
 [PASS] Sub-case 1 resolved across sparse matrices.
--- Phase 2: Preprocessing Architecture Formed ---
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['interest_rate' 'age']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
 [PASS] Sub-case 2 resolved across sparse matrices.
--- Phase

In [15]:
%%writefile week2_assignment_template.py
"""
M2 Coding Assignment: Enterprise Credit Risk Pipeline & Business-Driven Evaluation
Template Blueprint File — DO NOT MODIFY THE FUNCTION SIGNATURES OR DOCSTRINGS.
Students: Implement your code inside these functions.
"""

def load_and_map_data(filepath='credit_risk_dataset.csv'):
    """Phase 1: Ingestion, Column Standardization, and Stratified Splitting."""
    raise NotImplementedError("Not Implemented yet")

def build_preprocessing_architecture(X_train):
    """Phase 2: Constructing the Composite Dual-Pathway ColumnTransformer."""
    raise NotImplementedError("Not Implemented yet")

def train_master_pipeline(preprocessor, X_train, y_train):
    """Phase 3: Setting Algorithmic Guardrails and Compiling the Model Pipeline."""
    raise NotImplementedError("Not Implemented yet")

def calculate_tco(y_true, y_pred_labels):
    """Phase 5: Asymmetric Core Financial Matrix Calculator."""
    raise NotImplementedError("Not Implemented yet")

Writing week2_assignment_template.py


In [16]:
!python tester.py week2_assignment_template.py

EXECUTING GRADUATED MLOPS TEST HARNESS ON: week2_assignment_template.py

Checking Function: load_and_map_data()
--------------------------------------------------
 -> Result: [Not yet implemented] (Template intact, explicitly raises exception)

Checking Function: build_preprocessing_architecture()
--------------------------------------------------
 -> Result: [Not yet implemented] (Template intact, explicitly raises exception)

Checking Function: train_master_pipeline()
--------------------------------------------------
 -> Result: [Not yet implemented] (Template intact, explicitly raises exception)

Checking Function: calculate_tco()
--------------------------------------------------
 -> Result: [Not yet implemented] (Template intact, explicitly raises exception)

FINAL PERFORMANCE TALLY SUMMARY
Total number of test cases not implemented: 4
Total number of test cases passed:          0
Total number of test cases failed:          0


In [28]:
business_analysis_text = """ Business-Driven Evaluation & Threshold Optimization Analysis

Executive Summary: The Conflict Between Statistics and Dollars

In enterprise risk management, a model that performs exceptionally well on paper can still lead to catastrophic financial losses if its deployment strategy ignores business realities. Standard machine learning evaluations rely on a default probability threshold of 0.50, assuming a symmetric world where a false alarm carries the same weight as a missed threat.

In credit risk modeling, however, mistakes are fundamentally asymmetric. Failing to detect a defaulting borrower costs an organization the entire lost principal of the loan, whereas turning away a qualified applicant merely results in a minor operational or lost-opportunity cost. By transitioning from a standard statistical assessment to a Business-Driven Evaluation, we can align our machine learning pipeline with the company's bottom line—ultimately restructuring our decision boundaries to minimize Total Cost of Ownership (TCO).

The Statistical Baseline: A Flawed Victory

Evaluating the credit risk model at the standard statistical baseline (0.50 threshold) reveals a highly competent predictive engine. The system achieves a robust ROC-AUC (Receiver Operating Characteristic-Area Under the Curve) Score of 0.8850, proving that the underlying features and the trained gradient-boosted classifier (XGBClassifier) possess excellent discriminatory power.

--- Phase 4: Statistical Metrics (Standard 0.50 Threshold) ---

          precision    recall  f1-score   support
       0       0.93      0.82      0.87      5095
       1       0.55      0.77      0.64      1422
However, a deeper dive into the classification report exposes an operational vulnerability:

Class 1 (Default) Precision (55%): Out of all borrowers flagged by the model as a default risk, nearly half are actually safe. This leads to a high volume of false alarms.

Class 1 (Default) Recall (77%): The model captures 77% of actual defaults, which sounds strong statistically, but means that 23% of defaulting accounts are slipping through completely unnoticed.
When these percentages are processed through the corporate cost matrix—where a False Positive (FP) costs  500inlostopportunityandaFalseNegative(FN)costs 5,000 in unrecoverable principal loss—the financial impact of the baseline strategy becomes starkly clear.

Baseline TCO at 0.50 Threshold: $2,072,500  The Main Financial Cost

↳ False Positives (Type I): 915 | False Negatives (Type II): 323
 The breakdown showing exactly WHY it costs that much

At a 0.50 default cutoff, the enterprise experiences 915 false alarms and misses 323 defaults, culminating in a heavy Baseline TCO of $2,072,500. Even though the statistical accuracy sits comfortably at 81%, the organization faces severe financial exposure because the missed defaults are ten times more expensive than the false alarms.

Threshold Optimization: Simulating Risk Mitigation

To protect the enterprise's capital, we must actively shift our risk tolerance. The threshold optimization loop serves as a financial simulation, systematically lowering the probability requirement for a borrower to be flagged as a default risk. By lowering the threshold from 0.50 down to 0.05, we purposefully trade an increase in cheap false alarms for a drastic reduction in catastrophic missed defaults.

Threshold False Pos (FP) False Neg (FN) Total TCO
0.5 915 323  2,072,5000.41294229 1,792,000
0.3 1745 139  1,567,5000.2230182 1,560,500
0.15 2668 65  1,659,0000.1309943 1,764,500
0.05 3647 13 $1,888,500

As the cutoff falls, the financial dynamics shift across distinct phases:

The Aggressive Cost-Cutting Phase (0.50→0.30): Dropping the threshold initially yields massive dividends. Even though False Positives climb from 915 to 1,745, the number of missed defaults is slashed by more than half (from 323 down to 139). This aggressive reduction drops the TCO by over $500,000.

The Diminishing Returns Phase (0.20→0.05): Eventually, the strategy becomes overly sensitive. At a threshold of 0.05, the model captures almost every single default (only 13 are missed), but it flags an excessive 3,647 safe customers as risks. The massive operational overhead of these false alarms causes the TCO to climb back up to $1,888,500.
Conclusion: The Optimal Decision Boundary

By applying an automated optimization routine across our simulation ledger, the pipeline discovers the mathematical sweet spot for the enterprise's risk posture:

Optimal Business Decision Threshold discovered at: 0.2

Minimized Enterprise Financial Risk (TCO): $1,560,500

At the 0.20 probability threshold, the system balances both cost drivers perfectly. By telling the model to flag any applicant displaying even a modest 20% mathematical risk of default, the business accepts a higher volume of false alarms (2,301) in exchange for aggressively driving missed defaults down to just 82.

Financial Savings = Baseline TCO ( 2,072,500)−OptimizedTCO( 1,560,500) = $512,000

Deploying the model with this business-driven threshold yields an immediate cost reduction of $512,000 for the validation cohort alone. This analysis proves that the ultimate goal of enterprise data science is not to chase abstract academic metrics like accuracy, but to use statistical intelligence to engineer data-driven safeguards that maximize bottom-line profitability. """

# Define the absolute destination path inside your personal Google Drive space
#This saves it inside the root "My Drive" folder under a specific name
destination_path = '/content/drive/MyDrive/Enterprise_Credit_Risk_Analysis.txt'

# Execute the programmatic write operation (this goes to your Google Drive)
# with open(destination_path, 'w') as file: file.write(business_analysis_text.strip())

# print(f"✅ Success! Analysis write-up successfully saved to Google Drive at:\n{destination_path}")

# Save to a file in the content folder
with open('/content/Business_analysis_results.txt', 'w') as f:
    f.write(business_analysis_text)